In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def info_generale(df):
    """
    Stampa informazioni generali sul dataset, come tipo delle colonne e non-null count.
    """
    print("INFO GENERALE DATASET")
    print(df.info())

In [ ]:
def statistiche_descrittive(df):
    """
    Stampa statistiche descrittive (media, std, min, max, quartili) delle colonne numeriche.
    """
    print("STATISTICHE DESCRITTIVE")
    print(df.describe())

In [ ]:
def valori_mancanti(df):
    """
    Conta e stampa il numero di valori mancanti per ciascuna colonna del dataset.
    """
    print("VALORI MANCANTI")
    missing = df.isnull().sum()
    print(missing[missing > 0].sort_values(ascending=False))

In [ ]:
def distribuzione_target(df, target_col="VAL_REVENUES"):
    """
    Mostra l'istogramma e il boxplot del target specificato.
    """
    if target_col in df.columns:
        print(f"DISTRIBUZIONE {target_col}")
        plt.figure(figsize=(10,4))
        sns.histplot(df[target_col], kde=True, bins=50)
        plt.title(f"Distribuzione di {target_col}")
        plt.show()

        plt.figure()
        sns.boxplot(x=df[target_col])
        plt.title(f"Boxplot {target_col}")
        plt.show()

In [ ]:
def analisi_outlier(df, col="VAL_REVENUES"):
    """
    Identifica e stampa il numero di outlier nella colonna specificata usando IQR.
    """
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        outliers = df[
            (df[col] < q1 - 1.5 * iqr) |
            (df[col] > q3 + 1.5 * iqr)
        ]
        print("OUTLIER")
        print(f"Numero outlier: {len(outliers)}")

In [ ]:
def correlazioni(df):
    """
    Calcola e mostra la matrice di correlazione tra le colonne numeriche con heatmap.
    """
    print("MATRICE DI CORRELAZIONE")
    matrix = df.corr(numeric_only=True)
    plt.figure(figsize=(10,8))
    sns.heatmap(matrix, cmap="coolwarm", annot=False)
    plt.title("Correlation Matrix")
    plt.show()

In [ ]:
def analisi_temporale(df, revenue_col="VAL_REVENUES"):
    """
    Analizza l'andamento medio mensile delle revenue e lo visualizza come grafico.
    """
    if "ID_ORDER_DATE" in df.columns:
        df["ID_ORDER_DATE"] = pd.to_datetime(df["ID_ORDER_DATE"], errors="coerce")
        df["ANNO"] = df["ID_ORDER_DATE"].dt.year
        df["MESE"] = df["ID_ORDER_DATE"].dt.month

        trend = df.groupby("MESE")[revenue_col].mean().reset_index()
        plt.figure(figsize=(12,5))
        plt.plot(trend["MESE"], trend[revenue_col], marker="o", linewidth=2)
        plt.xticks(ticks=trend["MESE"], labels=trend["MESE"].astype(str).str.zfill(2))
        plt.title("Andamento medio mensile della revenue")
        plt.xlabel("Mese")
        plt.ylabel("Revenue media")
        plt.show()

In [ ]:
def analisi_categoriche(df, revenue_col="VAL_REVENUES", top_n=10):
    """
    Mostra i top N valori delle variabili categoriche rispetto alla revenue media.
    """
    cat_cols = df.select_dtypes(include="object").columns
    print("ANALISI VARIABILI CATEGORICHE")
    for col in cat_cols[:5]: 
        if revenue_col in df.columns:
            top = df.groupby(col)[revenue_col].mean().sort_values(ascending=False).head(top_n)
            plt.figure()
            top.plot(kind="bar")
            plt.title(f"{col} vs Revenue media")
            plt.xticks(rotation=45)
            plt.show()

In [ ]:
def statistiche_dettagliate(df):
    """
    Stampa statistiche dettagliate numeriche e tabelle di frequenze per ciascuna colonna numerica.
    """
    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        s = df[col].dropna()
        print(f"\nColonna: {col}")
        print(f"Media = {s.mean():.2f}, Min = {s.min()}, Max = {s.max()}, Varianza = {s.var(ddof=0):.2f}, Std = {s.std(ddof=0):.2f}, Mediana = {s.median()}, Moda = {s.mode().tolist()}")
        
        # Frequenze
        freq_ass = s.value_counts().sort_index()
        freq_rel = s.value_counts(normalize=True).sort_index()
        freq_rel_cum = freq_rel.cumsum()
        tabella = pd.DataFrame({
            "Modalità": freq_ass.index,
            "Freq Ass": freq_ass.values,
            "Freq Rel (%)": (freq_rel.values*100).round(1),
            "Freq Rel Cum (%)": (freq_rel_cum.values*100).round(1)
        })
        tabella.index = range(1,len(tabella)+1)
        tabella.index.name="Indice"
        print(tabella)

In [ ]:
def data_profiling(df, target_col="VAL_REVENUES"):
    """
    Esegue un profiling rapido del dataset chiamando le funzioni principali.
    """
    info_generale(df)
    valori_mancanti(df)
    statistiche_descrittive(df)
    distribuzione_target(df, target_col)
    correlazioni(df)
    analisi_categoriche(df, target_col)

In [ ]:
def main():
    
    df = pd.read_csv("SALES_OLAP.csv")
    # CONVERSIONE DATE perchè si legge da .csv
    
    df["ID_ORDER_DATE"] = pd.to_datetime(df["ID_ORDER_DATE"])
    df["ID_INVOICE_DATE"] = pd.to_datetime(df["ID_INVOICE_DATE"])
    # Analisi
    data_profiling(df, target_col="VAL_REVENUES")
    analisi_outlier(df)
    statistiche_dettagliate(df)
    analisi_temporale(df)

    return df

if __name__ == "__main__":
    main()